In [30]:
import os
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torchvision.models as models
import torchvision

In [4]:
# 1. Cihaz (Device) Tespiti
device = torch.device('mps' if torch.mps.is_available() else 'cpu')

In [8]:
# 2. Hiperparametreler ve Dosya Yolları
BATCH_SIZE = 32
NUM_WORKERS = os.cpu_count()

data_path = Path('')
image_path = data_path / 'imagewoof200'
train_dir = image_path / 'train'
test_dir = image_path / 'test'

In [10]:
# 3. Model Ağırlıkları ve Otomatik Dönüşümler (Transforms)
weights = models.GoogLeNet_Weights.DEFAULT
auto_transforms = weights.transforms()

In [12]:
# 4. DataLoaders Oluşturma Fonksiyonu ve Yükleyicilerin Hazırlanması
def create_dataloader(
    train_dir: Path,
    test_dir: Path,
    transforms: transforms.Compose,
    batch_size: int = BATCH_SIZE,
    num_workers: int = NUM_WORKERS
):
    train_data = datasets.ImageFolder(train_dir, transform=transforms)
    test_data = datasets.ImageFolder(test_dir, transform=transforms)

    class_names = train_data.classes

    # MPS üzerinde pin_memory desteklenmediği için pin_memory=False tutuldu
    train_dataloader = DataLoader(
        dataset=train_data,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=False
    )
    
    test_dataloader = DataLoader(
        dataset=test_data,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=False
    )

    return train_dataloader, test_dataloader, class_names

train_dataloader, test_dataloader, class_names = create_dataloader(
    train_dir=train_dir,
    test_dir=test_dir,
    transforms=auto_transforms
)

In [14]:
# 5. Model Mimarisi, Katman Dondurma ve Sınıflandırıcı (FC) Güncellemesi
model = models.googlenet(weights=weights)

In [16]:
# Omurga katmanlarının parametrelerini dondurma (Feature Extraction)
for param in model.parameters():
    param.requires_grad = False

In [18]:
# Son katmanı kendi sınıf sayımıza göre yeniden yapılandırma
out_features = len(class_names)
model.fc = nn.Linear(in_features=1024, out_features=out_features)

In [20]:
# Modeli hedef cihaza taşıma
model = model.to(device)

In [22]:
# 6. Kayıp Fonksiyonu (Loss) ve Optimizer
loss_fn = nn.CrossEntropyLoss()
# Sadece eğitilebilir (requires_grad=True olan) fc katmanı ağırlıklarını optimizere veriyoruz
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)

In [24]:
# 7. Eğitim ve Test Adım Fonksiyonları
def train_step(model: torch.nn.Module,
               dataloader: DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               device: torch.device):
    model.train()
    train_loss, train_acc = 0, 0

    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        y_pred = model(X)
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item() / len(y_pred)

    return train_loss / len(dataloader), train_acc / len(dataloader)


def test_step(model: torch.nn.Module,
              dataloader: DataLoader,
              loss_fn: torch.nn.Module,
              device: torch.device):
    model.eval()
    test_loss, test_acc = 0, 0

    with torch.inference_mode():
        for batch, (X, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)

            test_pred_logits = model(X)
            loss = loss_fn(test_pred_logits, y)
            test_loss += loss.item()

            test_pred_labels = test_pred_logits.argmax(dim=1)
            test_acc += ((test_pred_labels == y).sum().item() / len(test_pred_labels))

    return test_loss / len(dataloader), test_acc / len(dataloader)


def train(model: torch.nn.Module,
          train_dataloader: DataLoader,
          test_dataloader: DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module,
          epochs: int,
          device: torch.device):

    results = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

    for epoch in range(epochs):
        train_loss, train_acc = train_step(
            model=model,
            dataloader=train_dataloader,
            loss_fn=loss_fn,
            optimizer=optimizer,
            device=device
        )
        test_loss, test_acc = test_step(
            model=model,
            dataloader=test_dataloader,
            loss_fn=loss_fn,
            device=device
        )

        print(
            f"Epoch: {epoch + 1} | "
            f"train_loss: {train_loss:.4f} | "
            f"train_acc: {train_acc:.4f} | "
            f"test_loss: {test_loss:.4f} | "
            f"test_acc: {test_acc:.4f}"
        )

        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)

    return results

In [26]:
# 8. Eğitimin Başlatılması
results = train(
    model=model,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    epochs=5,
    device=device
)

Epoch: 1 | train_loss: 1.3052 | train_acc: 0.6949 | test_loss: 0.6482 | test_acc: 0.8854
Epoch: 2 | train_loss: 0.5801 | train_acc: 0.8700 | test_loss: 0.4563 | test_acc: 0.8833
Epoch: 3 | train_loss: 0.4418 | train_acc: 0.8884 | test_loss: 0.3856 | test_acc: 0.9021
Epoch: 4 | train_loss: 0.3812 | train_acc: 0.9003 | test_loss: 0.3715 | test_acc: 0.8938
Epoch: 5 | train_loss: 0.3457 | train_acc: 0.9072 | test_loss: 0.3512 | test_acc: 0.9021


In [32]:
from typing import List, Tuple
import matplotlib.pyplot as plt
from PIL import Image

def transform_predict(model: torch.nn.Module,
                        image_path: str,
                        class_names: List[str],
                        image_size: Tuple[int, int] = (224, 224),
                        transform: torchvision.transforms = None,
                        device: torch.device=device):

    img = Image.open(image_path)

    if transform is not None:
        image_transform = transform
    else:
        image_transform = transforms.Compose([
            transforms.Resize(image_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])


    model.to(device)
    model.eval()
    with torch.inference_mode():
        transformed_image = image_transform(img).unsqueeze(dim=0)
        target_image_pred = model(transformed_image.to(device))
        target_image_pred_probs = torch.softmax(target_image_pred, dim=1)
        target_image_pred_label = torch.argmax(target_image_pred_probs, dim=1)

    plt.figure()
    plt.imshow(img)
    plt.title(f"Pred: {class_names[target_image_pred_label]} | Prob: {target_image_pred_probs.max():.2f}")
    plt.axis(False)

In [ ]:
custom_image_path = data_path / "hedef_gorsel.jpeg"
transform_predict(model=model,
                    image_path=custom_image_path,
                    class_names=class_names)